# Random Forest — Regresor

**Repositorio:** `autodidact/both/random_forest/`

---

## 1. Idea General e Intuición

El **Random Forest Regresor** usa el mismo mecanismo que el clasificador (bagging + selección aleatoria de features), pero en lugar de votar por una clase, **promedia las predicciones** de todos los árboles.

**Predicción final:**
$$\hat{y} = \frac{1}{T} \sum_{t=1}^{T} \hat{y}_t(\mathbf{x})$$

Donde $\hat{y}_t(\mathbf{x})$ es la predicción del árbol $t$ para la entrada $\mathbf{x}$.

**Ventaja clave sobre un árbol único:** El promedio de $T$ estimadores reduce la varianza:
$$\text{Var}(\bar{Y}) \approx \rho \sigma^2 + \frac{(1-\rho)}{T} \sigma^2$$

La aleatoriedad en features reduce la correlación $\rho$ entre árboles, lo que hace que el segundo término caiga con $T$.

## 2. Fundamento Matemático

### 2.1 Criterio de División en Regresión

En regresión los nodos se dividen minimizando la **varianza** (MSE) de las salidas:

$$\Delta \text{MSE} = \text{MSE}(t) - \frac{n_L}{n}\text{MSE}(t_L) - \frac{n_R}{n}\text{MSE}(t_R)$$

Donde:
$$\text{MSE}(t) = \frac{1}{n_t}\sum_{i \in t}(y_i - \bar{y}_t)^2$$

### 2.2 Métricas de Evaluación para Regresión

| Métrica | Fórmula | Interpretación |
|---|---|---|
| **MSE** | $\frac{1}{n}\sum(y_i - \hat{y}_i)^2$ | Penaliza errores grandes (sensible a outliers) |
| **RMSE** | $\sqrt{\text{MSE}}$ | Mismas unidades que $y$ |
| **MAE** | $\frac{1}{n}\sum\|y_i - \hat{y}_i\|$ | Robusto a outliers |
| **$R^2$** | $1 - \frac{\sum(y_i-\hat{y}_i)^2}{\sum(y_i-\bar{y})^2}$ | Proporción de varianza explicada; 1=perfecto |
| **MAPE** | $\frac{100}{n}\sum\frac{\|y_i-\hat{y}_i\|}{y_i}$ | Error relativo (%) |

**OOB Score en regresión:** reporta $R^2$ calculado sobre las muestras out-of-bag.

### 2.3 Intervalo de Predicción Empírico

RF no produce intervalos de confianza analíticos, pero se puede estimar la **dispersión de predicciones entre árboles**:

$$\text{Incertidumbre}(\mathbf{x}) \approx \text{std}\left(\{\hat{y}_1(\mathbf{x}), \hat{y}_2(\mathbf{x}), \ldots, \hat{y}_T(\mathbf{x})\}\right)$$

## 3. ¿Cuándo Usarlo y Cuándo NO?

### ✅ Úsalo cuando:
- Tienes relaciones **no lineales complejas** entre features y target.
- El dataset tiene **outliers** en las features (RF es robusto a ellos).
- Necesitas **importancia de features** sin mucho preprocesamiento.
- Tienes features **heterogéneas** (mezcla de escalas, tipos).

### ❌ NO lo uses cuando:
- Necesitas **extrapolación** (RF no puede predecir más allá del rango de entrenamiento).
- El target tiene **tendencias temporales** (series de tiempo requieren modelos especiales).
- El dataset es **muy pequeño** (riesgo de sobreajuste al bootstrap).
- La **interpretabilidad causal** es requisito (usa modelos lineales).

## 5. Problemas Específicos del Regresor

### 5.1 No Extrapolación
RF predice el promedio de hojas entrenadas. Si el test tiene valores fuera del rango de entrenamiento, RF **regresa al promedio de los extremos**, no extrapola.

### 5.2 Suavizado Excesivo
Para $T$ grande, el promedio puede suavizar demasiado los picos y valles del target.

### 5.3 Importancia MDI Sesgada en Regresión
Igual que en clasificación, features con alta cardinalidad obtienen importancia inflada artificialmente. Usar `permutation_importance`.

### 5.4 Outliers en el Target
A diferencia de los outliers en features (a los que RF es robusto), **outliers en el target** afectan el criterio MSE de división. Alternativa: usar criterio `criterion='absolute_error'`.

## 6. Mejora de Resultados

| Parámetro | Valor típico | Efecto |
|---|---|---|
| `n_estimators` | 100-500 | Más árboles → menor varianza |
| `max_features` | `'sqrt'` o `1/3` | Menor → menos correlación entre árboles |
| `max_depth` | `None` o 10-30 | `None` = árboles completos |
| `min_samples_leaf` | 1-10 | Mayor → más suavizado, menos sobreajuste |
| `criterion` | `'squared_error'` / `'absolute_error'` | MAE más robusto a outliers en target |

In [ ]:
# =============================================================================
# SETUP
# =============================================================================
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.datasets import fetch_california_housing, load_diabetes
from sklearn.model_selection import train_test_split, cross_val_score, GridSearchCV
from sklearn.ensemble import RandomForestRegressor
from sklearn.inspection import permutation_importance, PartialDependenceDisplay
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score

import warnings
warnings.filterwarnings('ignore')

plt.style.use('seaborn-v0_8-darkgrid')
SEED = 42
print('✅ Librerías cargadas.')

---
## 7. Regresión — California Housing

In [ ]:
# =============================================================================
# CARGA DEL DATASET
# California Housing: 20640 muestras, 8 features, target = precio mediano de casas
# =============================================================================
housing = fetch_california_housing(as_frame=True)
X, y = housing.data, housing.target

print(f'Shape: {X.shape}')
print(f'Target — Media: {y.mean():.3f} | Std: {y.std():.3f} | Rango: [{y.min():.2f}, {y.max():.2f}]')
housing.frame.describe().round(3)

In [ ]:
# =============================================================================
# TRAIN/TEST SPLIT Y MODELO BASE
# RF no requiere escalado de features
# =============================================================================
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=SEED
)

rfr = RandomForestRegressor(
    n_estimators=200,
    max_features='sqrt',         # Para regresión también funciona bien sqrt(d)
    max_depth=None,
    min_samples_leaf=2,          # Mínimo 2 muestras por hoja → reduce sobreajuste
    oob_score=True,
    criterion='squared_error',
    random_state=SEED,
    n_jobs=-1
)
rfr.fit(X_train, y_train)
y_pred = rfr.predict(X_test)

rmse = np.sqrt(mean_squared_error(y_test, y_pred))
mae  = mean_absolute_error(y_test, y_pred)
r2   = r2_score(y_test, y_pred)

print(f'OOB R²  : {rfr.oob_score_:.4f}  ← estimación gratuita sin CV')
print(f'Test R² : {r2:.4f}')
print(f'RMSE    : {rmse:.4f} (× $100k)')
print(f'MAE     : {mae:.4f} (× $100k)')

In [ ]:
# =============================================================================
# VISUALIZACIÓN 1: Predichos vs Reales
# La diagonal perfecta indica predicción perfecta
# =============================================================================
fig, axes = plt.subplots(1, 2, figsize=(13, 5))

# Scatter: predichos vs reales
axes[0].scatter(y_test, y_pred, alpha=0.3, s=10, color='steelblue')
lim = [y_test.min(), y_test.max()]
axes[0].plot(lim, lim, 'r--', lw=2, label='Predicción perfecta')
axes[0].set_xlabel('Valores Reales', fontsize=11)
axes[0].set_ylabel('Predicciones', fontsize=11)
axes[0].set_title(f'Predichos vs Reales\n$R^2$ = {r2:.4f}', fontsize=12, fontweight='bold')
axes[0].legend()

# Distribución de residuos
residuals = y_test.values - y_pred
axes[1].hist(residuals, bins=60, color='steelblue', alpha=0.7, edgecolor='white')
axes[1].axvline(0, color='crimson', linestyle='--', lw=2, label='Error=0')
axes[1].set_xlabel('Residuo (Real - Predicho)', fontsize=11)
axes[1].set_ylabel('Frecuencia', fontsize=11)
axes[1].set_title(f'Distribución de Residuos\nMedia={residuals.mean():.4f}',
                  fontsize=12, fontweight='bold')
axes[1].legend()

plt.suptitle('Random Forest Regresor — California Housing',
             fontsize=14, fontweight='bold', y=1.02)
plt.tight_layout()
plt.show()

In [ ]:
# =============================================================================
# VISUALIZACIÓN 2: Importancia de Features (MDI vs Permutation)
# =============================================================================
feat_names = housing.feature_names

# MDI
mdi_imp = pd.Series(rfr.feature_importances_, index=feat_names).sort_values(ascending=False)

# Permutation Importance (más honesto)
perm_res = permutation_importance(rfr, X_test, y_test, n_repeats=15,
                                   random_state=SEED, n_jobs=-1)
perm_imp = pd.Series(perm_res.importances_mean, index=feat_names).sort_values(ascending=False)

fig, axes = plt.subplots(1, 2, figsize=(13, 5))

axes[0].barh(mdi_imp.index[::-1], mdi_imp.values[::-1], color='steelblue', alpha=0.8)
axes[0].set_title('Importancia MDI (Gini / MSE)', fontsize=12, fontweight='bold')
axes[0].set_xlabel('Importancia media')

# Para permutation, añadir barras de error (std)
perm_std = pd.Series(perm_res.importances_std, index=feat_names).loc[perm_imp.index]
axes[1].barh(perm_imp.index[::-1], perm_imp.values[::-1],
             xerr=perm_std.values[::-1], color='seagreen', alpha=0.8, capsize=4)
axes[1].set_title('Permutation Importance (± std)', fontsize=12, fontweight='bold')
axes[1].set_xlabel('Reducción media de R²')

plt.suptitle('Importancia de Features — California Housing',
             fontsize=13, fontweight='bold')
plt.tight_layout()
plt.show()

In [ ]:
# =============================================================================
# VISUALIZACIÓN 3: Partial Dependence Plots (PDP)
# Muestra el efecto marginal de cada feature sobre la predicción,
# manteniendo las demás constantes en su media.
# =============================================================================
features_to_plot = ['MedInc', 'AveRooms', 'Latitude', 'Longitude']
fig, ax = plt.subplots(figsize=(14, 5))

PartialDependenceDisplay.from_estimator(
    rfr, X_train, features=features_to_plot,
    feature_names=feat_names,
    n_cols=4, ax=ax,
    grid_resolution=50
)
fig.suptitle('Partial Dependence Plots — California Housing',
             fontsize=13, fontweight='bold')
plt.tight_layout()
plt.show()

In [ ]:
# =============================================================================
# INCERTIDUMBRE DE PREDICCIÓN (dispersión entre árboles)
# Calcula std de las predicciones de todos los árboles individuales
# para cuantificar la incertidumbre epistémica del modelo.
# =============================================================================
# Predicciones de cada árbol individual
all_tree_preds = np.array([tree.predict(X_test) for tree in rfr.estimators_])
pred_std = all_tree_preds.std(axis=0)   # std entre árboles por muestra

# Tomar una muestra de 200 puntos para visualizar
idx = np.random.RandomState(SEED).choice(len(y_test), 200, replace=False)
y_test_arr = y_test.values[idx]
y_pred_arr = y_pred[idx]
std_arr    = pred_std[idx]
order      = np.argsort(y_test_arr)

fig, ax = plt.subplots(figsize=(13, 5))
ax.errorbar(range(200), y_pred_arr[order], yerr=2*std_arr[order],
            fmt='none', alpha=0.3, color='steelblue', label='±2σ entre árboles')
ax.scatter(range(200), y_test_arr[order], s=12, color='crimson',
           zorder=5, label='Valor real')
ax.scatter(range(200), y_pred_arr[order], s=8, color='steelblue',
           zorder=4, alpha=0.7, label='Predicción RF')
ax.set_xlabel('Muestras (ordenadas por valor real)', fontsize=11)
ax.set_ylabel('Precio (×$100k)', fontsize=11)
ax.set_title('Incertidumbre de Predicción: Dispersión entre Árboles',
             fontsize=13, fontweight='bold')
ax.legend()
plt.tight_layout()
plt.show()

In [ ]:
# =============================================================================
# HYPERPARAMETER TUNING + CURVA DE VALIDACIÓN: max_depth
# =============================================================================
depth_values = [3, 5, 8, 12, 18, 25, None]
cv_r2_scores = []

for depth in depth_values:
    rf_tmp = RandomForestRegressor(n_estimators=100, max_depth=depth,
                                    max_features='sqrt', random_state=SEED, n_jobs=-1)
    sc = cross_val_score(rf_tmp, X_train, y_train, cv=5, scoring='r2')
    cv_r2_scores.append(sc.mean())
    label = str(depth) if depth else 'None'
    print(f'max_depth={label:>4s}  →  CV R²: {sc.mean():.4f} ± {sc.std():.4f}')

fig, ax = plt.subplots(figsize=(9, 4))
x_labels = [str(d) if d else 'None' for d in depth_values]
ax.plot(x_labels, cv_r2_scores, 'o-', color='steelblue', lw=2, ms=8)
ax.set_xlabel('max_depth', fontsize=12)
ax.set_ylabel('CV R² (5-fold)', fontsize=12)
ax.set_title('Efecto de max_depth en el Regresor', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.show()

---
## 8. Resumen Comparativo Clasificador vs Regresor

| Aspecto | Clasificador | Regresor |
|---|---|---|
| **Combinación** | Votación mayoritaria | Promedio aritmético |
| **Criterio nodo** | Gini / Entropía | MSE / MAE |
| **`max_features`** | `sqrt(d)` | `d/3` o `sqrt(d)` |
| **OOB Score** | Accuracy | $R^2$ |
| **Limitación clave** | Clases desbalanceadas | No extrapola |

### ✅ Checklist Regresor:
- [ ] ¿Verifiqué que el target no requiere extrapolación?
- [ ] ¿Grafiqué Predichos vs Reales y la distribución de residuos?
- [ ] ¿Usé `oob_score=True` durante el desarrollo?
- [ ] ¿Comparé MDI vs Permutation Importance?
- [ ] ¿Consideré `criterion='absolute_error'` si hay outliers en el target?

---
*Ver notebook de clasificación: `rf_clasificador.ipynb`*